# What should Ubisoft's next game be?

A market study of the **55 691 games** listed on Steam, run for Ubisoft's studio leadership.

Jedha *Full Stack Data Scientist* — **Block 2, Big Data Project**. PySpark on Databricks.

---

Ubisoft wants to launch a new game and asked for a global reading of the Steam marketplace
before the concept is locked. The brief lists a dozen questions on three levels — the market as
a whole, genres, and platforms. This notebook answers them in that order, and each level closes
on the one thing it changes in the product brief: **genre, price, platforms, languages, release
window, age rating.**

Section 6 collects those six decisions. Section 7 says what this dataset cannot decide.

## The data

`steam_game_output.json` — a 61 MB JSON array, one object per game, `{"id": ..., "data": {...}}`,
served from `s3://full-stack-bigdata-datasets/Big_Data/Project_Steam/`. It is a **SteamSpy
snapshot** and the newest release in it is **11 November 2022**, so every count for 2022 is
partial and no game released after that date exists here.

Three of the 22 fields are nested — `tags` (a tag → number-of-votes object), `platforms`
(three booleans) and `categories` (a list) — which is what makes the file semi-structured and
what `explode()` and `getField()` are for.

## 0. Environment

The notebook is written for **Databricks**, and falls back to a local Spark session so the code
can be re-run outside a workspace. Everything below this cell is identical in both cases,
including the `display()` calls that drive Databricks' visualisation tool.

In [1]:
import os

IS_DATABRICKS = "DATABRICKS_RUNTIME_VERSION" in os.environ
SOURCE_URL = ("https://full-stack-bigdata-datasets.s3.amazonaws.com"
              "/Big_Data/Project_Steam/steam_game_output.json")

if IS_DATABRICKS:
    # Unity Catalog volume: the file is fetched once, then read from storage.
    spark.sql("CREATE VOLUME IF NOT EXISTS workspace.default.steam")
    DATA_PATH = "/Volumes/workspace/default/steam/steam_game_output.json"
    if not os.path.exists(DATA_PATH):
        import urllib.request
        urllib.request.urlretrieve(SOURCE_URL, DATA_PATH)
else:
    # Local run: build the session by hand and emulate Databricks' display().
    from pyspark.sql import SparkSession, DataFrame
    from IPython.display import display as _ipython_display

    os.environ.setdefault("SPARK_LOCAL_IP", "127.0.0.1")
    spark = (SparkSession.builder.appName("steam")
             .master("local[*]")
             .config("spark.driver.memory", "10g")
             .config("spark.sql.session.timeZone", "UTC")
             .config("spark.ui.showConsoleProgress", "false")
             .getOrCreate())
    spark.sparkContext.setLogLevel("ERROR")

    def display(x, n=1000):
        _ipython_display(x.limit(n).toPandas() if isinstance(x, DataFrame) else x)

    DATA_PATH = "data/steam_game_output.json"

from pyspark.sql import functions as F, Window
from pyspark.sql.types import (StructType, StructField, StringType, LongType,
                               BooleanType, ArrayType, MapType)

print("Spark", spark.version, "| Databricks" if IS_DATABRICKS else "| local", "|", DATA_PATH)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | local | data/steam_game_output.json


## 1. Reading a semi-structured file

The whole array sits on **one line**, so `multiLine` has to be on: without it Spark splits the
file on newlines and finds a single unparseable record.

### 1.1 Why schema inference is not enough

Let Spark infer the schema and `tags` — an object whose *keys are the tag names* — becomes a
struct with one column per tag seen anywhere in the file. Inference also has to scan the 61 MB
twice, and it silently picks a type for `required_age`, a field that holds both numbers and
strings.

In [2]:
inferred = spark.read.option("multiLine", True).json(DATA_PATH)

tags_field = inferred.schema["data"].dataType["tags"].dataType
print("columns Spark invented for `tags`:", len(tags_field.fields))
print("first five:", [f.name for f in tags_field.fields[:5]])

columns Spark invented for `tags`: 441
first five: ['1980s', "1990's", '2.5D', '2D', '2D Fighter']


### 1.2 An explicit schema instead

Declaring `tags` as `MAP<STRING, BIGINT>` turns those 441 phantom columns into one map column
that `explode()` can open into (tag, votes) rows. `required_age` is declared `STRING` and parsed
later, where the parsing rule is visible.

In [3]:
DATA_SCHEMA = StructType([
    StructField("appid", LongType()),
    StructField("name", StringType()),
    StructField("short_description", StringType()),
    StructField("developer", StringType()),
    StructField("publisher", StringType()),
    StructField("genre", StringType()),                       # comma-separated
    StructField("tags", MapType(StringType(), LongType())),   # tag -> community votes
    StructField("type", StringType()),
    StructField("categories", ArrayType(StringType())),
    StructField("owners", StringType()),                      # bucketed range, e.g. "0 .. 20,000"
    StructField("positive", LongType()),
    StructField("negative", LongType()),
    StructField("price", StringType()),                       # US cents, as a string
    StructField("initialprice", StringType()),
    StructField("discount", StringType()),                    # percent, as a string
    StructField("ccu", LongType()),                           # peak concurrent users
    StructField("languages", StringType()),                   # comma-separated
    StructField("platforms", StructType([
        StructField("windows", BooleanType()),
        StructField("mac", BooleanType()),
        StructField("linux", BooleanType()),
    ])),
    StructField("release_date", StringType()),
    StructField("required_age", StringType()),
    StructField("website", StringType()),
    StructField("header_image", StringType()),
])

SCHEMA = StructType([
    StructField("id", StringType()),
    StructField("data", DATA_SCHEMA),
])

raw = (spark.read.schema(SCHEMA).option("multiLine", True).json(DATA_PATH)
       .select("id", "data.*"))          # flatten the nested `data` struct

print(f"{raw.count():,} games x {len(raw.columns)} columns")
raw.printSchema()

55,691 games x 23 columns
root
 |-- id: string (nullable = true)
 |-- appid: long (nullable = true)
 |-- name: string (nullable = true)
 |-- short_description: string (nullable = true)
 |-- developer: string (nullable = true)
 |-- publisher: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- tags: map (nullable = true)
 |    |-- key: string
 |    |-- value: long (valueContainsNull = true)
 |-- type: string (nullable = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- owners: string (nullable = true)
 |-- positive: long (nullable = true)
 |-- negative: long (nullable = true)
 |-- price: string (nullable = true)
 |-- initialprice: string (nullable = true)
 |-- discount: string (nullable = true)
 |-- ccu: long (nullable = true)
 |-- languages: string (nullable = true)
 |-- platforms: struct (nullable = true)
 |    |-- windows: boolean (nullable = true)
 |    |-- mac: boolean (nullable = true)
 |    |-- linux: boolean (nulla

### 1.3 The map opens

`explode()` on the `tags` map is what gives every later tag question a row-per-tag table.

In [4]:
display(
    raw.filter(F.col("name") == "Counter-Strike")
       .select("name", F.explode("tags").alias("tag", "votes"))
       .orderBy(F.desc("votes")).limit(8)
)

,name,tag,votes
0,Counter-Strike,Action,5426
1,Counter-Strike,FPS,4831
2,Counter-Strike,Multiplayer,3392
3,Counter-Strike,Shooter,3353
4,Counter-Strike,Classic,2784
5,Counter-Strike,Team-Based,1864
6,Counter-Strike,First-Person,1707
7,Counter-Strike,Competitive,1607


## 2. Cleaning

Nine rules. Each one is stated with the number of rows it touches, so nothing is silently
dropped or rewritten.

### 2.1 Keys and scope

`id` duplicates `appid`, and every `appid` is unique — there is nothing to deduplicate. One row
is not a game.

In [5]:
print("rows where id != appid:", raw.filter(F.col("id") != F.col("appid").cast("string")).count())
print("distinct appid:", raw.select("appid").distinct().count(), "of", raw.count(), "rows")
display(raw.groupBy("type").count())

rows where id != appid: 0


distinct appid: 55691 of 55691 rows


,type,count
0,hardware,1
1,game,55690


### 2.2 Money

`price`, `initialprice` and `discount` are strings. Prices are US cents, so a `999` is $9.99.
`price` is the current price and `initialprice` the list price: the two differ on exactly the
2 518 rows that carry a discount, so the field is internally consistent and `initialprice` is
the one to use for anything about a game's positioning.

In [6]:
games = (raw.filter(F.col("type") == "game")
         .withColumn("price_usd", F.col("price").cast("int") / 100)
         .withColumn("initial_price_usd", F.col("initialprice").cast("int") / 100)
         .withColumn("discount_pct", F.col("discount").cast("int"))
         .withColumn("is_free", F.col("price").cast("int") == 0))

display(games.select(
    F.round(F.min("price_usd"), 2).alias("min_price"),
    F.round(F.max("price_usd"), 2).alias("max_price"),
    F.sum(F.col("is_free").cast("int")).alias("free_games"),
    F.sum((F.col("discount_pct") > 0).cast("int")).alias("discounted"),
    F.sum((F.col("price") != F.col("initialprice")).cast("int")).alias("price_below_list"),
))

,min_price,max_price,free_games,discounted,price_below_list
0,0.0,999.0,7779,2518,2518


### 2.3 Release dates

Four shapes in one column: `2020/06/18`, `2020/06/8` (one-digit day), `2015/09` (no day at all)
and empty. Matching each shape before parsing keeps the 99 empty strings as an explicit `NULL`
instead of a silent failure.

In [7]:
games = (games
    .withColumn("release_date_parsed",
        F.when(F.col("release_date").rlike(r"^\d{4}/\d{1,2}/\d{1,2}$"),
               F.to_date("release_date", "yyyy/M/d"))
         .when(F.col("release_date").rlike(r"^\d{4}/\d{1,2}$"),
               F.to_date("release_date", "yyyy/M")))
    .withColumn("release_year", F.year("release_date_parsed"))
    .withColumn("release_month", F.month("release_date_parsed")))

display(games.select(
    F.min("release_date_parsed").alias("first_release"),
    F.max("release_date_parsed").alias("last_release"),
    F.sum(F.col("release_date_parsed").isNull().cast("int")).alias("unparseable"),
))

,first_release,last_release,unparseable
0,1997-06-30,2022-11-11,99


### 2.4 Comma-separated lists

`genre` and `languages` pack several values into one string. Splitting on the comma is not quite
enough: four rows carry a parenthetical note (`English (full audio)`), two a stray semicolon,
and one lists English twice. Stripping the noise, trimming and deduplicating gives clean arrays —
and note that `Spanish - Spain` and `Design & Illustration` mean the separator can only ever be
the comma.

In [8]:
def split_list(column):
    """Comma-separated string -> trimmed, deduplicated array, parenthetical notes removed."""
    parts = F.split(F.regexp_replace(F.col(column), r"\([^)]*\)|[;*]", ""), ",")
    return F.array_distinct(F.array_remove(F.transform(parts, lambda x: F.trim(x)), ""))

games = (games
    .withColumn("genres", split_list("genre"))
    .withColumn("languages_list", split_list("languages"))
    .withColumn("n_genres", F.size("genres"))
    .withColumn("n_languages", F.size("languages_list")))

display(games.filter(F.col("languages").contains("(")).select("name", "languages", "languages_list"))

,name,languages,languages_list
0,Ninja Reflex: Steamworks Edition,"English, French, German, Italian, Spanish - Sp...","[English, French, German, Italian, Spanish - S..."
1,The Witcher: Enhanced Edition Director's Cut,"English, French, German, Spanish - Spain, Ital...","[English, French, German, Spanish - Spain, Ita..."
2,Wallace & Gromit’s Grand Adventures,"English (full audio), French, German, Italian,...","[English, French, German, Italian, Spanish - S..."
3,Marble Masters: The Pit,"English, French, German, Spanish - Spain, Czec...","[English, French, German, Spanish - Spain, Cze..."


### 2.5 Publisher names

Ubisoft appears under five spellings, two of which differ only by a trademark sign or four
trailing tabs. Trimming whitespace and stripping `®`/`™` rewrites 271 rows; it does **not**
merge `Ubisoft` with `Ubisoft Entertainment`, and it should not — deciding that two different
company names are the same firm is a judgement call, not a cleaning rule. The publisher counts in
section 3.1 are therefore a floor, not an exact figure.

The field also holds co-publisher lists (`Team17, NEXT Studios`), but 3 215 rows contain a comma
and most of them are `Ltd.`-style suffixes, so splitting on it would create more noise than it
removes. The string is kept whole.

In [9]:
def normalise_name(column):
    cleaned = F.trim(F.regexp_replace(F.regexp_replace(F.col(column), r"[®™]", ""), r"\s+", " "))
    return F.when(cleaned != "", cleaned)

games = (games
    .withColumn("publisher_clean", normalise_name("publisher"))
    .withColumn("developer_clean", normalise_name("developer")))

print("distinct publisher strings:", games.select("publisher").distinct().count(),
      "-> after normalisation:", games.select("publisher_clean").distinct().count())

display(games.filter(F.col("publisher").rlike("^Ubisoft"))
             .groupBy("publisher", "publisher_clean").count().orderBy(F.desc("count")))

distinct publisher strings: 29966 -> after normalisation: 29825


,publisher,publisher_clean,count
0,Ubisoft,Ubisoft,127
1,Ubisoft Entertainment,Ubisoft Entertainment,5
2,Ubisoft Entertainment\t\t\t\t,Ubisoft Entertainment,1
3,Ubisoft®,Ubisoft,1
4,Ubisoft - San Francisco,Ubisoft - San Francisco,1


### 2.6 Owners

SteamSpy does not publish a sales figure, it publishes a bracket: `"10,000,000 .. 20,000,000"`.
Two regexes give the bounds, and the midpoint stands in for the value. **68% of the catalogue
sits in the bottom bracket**, so the median of that midpoint is `10,000` almost everywhere and is
useless as a comparison — the rest of the notebook uses two other measures instead: the **median
review count**, and the **share of games above 100 000 owners** (the "breakout rate").

In [10]:
owners_digits = F.regexp_replace(F.col("owners"), ",", "")
games = (games
    .withColumn("owners_min", F.regexp_extract(owners_digits, r"^(\d+)", 1).cast("long"))
    .withColumn("owners_max", F.regexp_extract(owners_digits, r"\.\.\s*(\d+)$", 1).cast("long")))
games = games.withColumn("owners_mid", (F.col("owners_min") + F.col("owners_max")) / 2)

display(games.groupBy("owners", "owners_min", "owners_max").count().orderBy("owners_min"))

,owners,owners_min,owners_max,count
0,"0 .. 20,000",0,20000,38072
1,"20,000 .. 50,000",20000,50000,7285
2,"50,000 .. 100,000",50000,100000,3695
3,"100,000 .. 200,000",100000,200000,2519
4,"200,000 .. 500,000",200000,500000,2162
5,"500,000 .. 1,000,000",500000,1000000,932
6,"1,000,000 .. 2,000,000",1000000,2000000,526
7,"2,000,000 .. 5,000,000",2000000,5000000,335
8,"5,000,000 .. 10,000,000",5000000,10000000,97
9,"10,000,000 .. 20,000,000",10000000,20000000,41


### 2.7 Required age

The field mixes integers and strings, and its odd values are `"MA 15+"`, `"21+"`, `"7+"`, `"35"`
and `"180"` (four times). Pulling the first number out handles the `+` suffixes; anything outside
0-21 is not an age rating and becomes `NULL` — five rows.

That fixes the parsing but not the field: **it is 0 for 98.8% of the catalogue**, because Steam
gates mature content through its own content descriptors rather than this legacy attribute.
Section 3.5 answers the brief's question about age-restricted games from the community tags
instead.

In [11]:
age = F.regexp_extract(F.col("required_age"), r"(\d+)", 1).cast("int")
games = games.withColumn("age_rating", F.when(age.between(0, 21), age))

display(games.groupBy("required_age", "age_rating").count().orderBy(F.desc("count")).limit(25))

,required_age,age_rating,count
0,0,0.0,55029
1,15,15.0,264
2,18,18.0,223
3,17,17.0,38
4,16,16.0,38
5,12,12.0,32
6,13,13.0,26
7,14,14.0,10
8,10,10.0,7
9,6,6.0,4


### 2.8 Reviews

`positive` and `negative` are review counts, not scores. The raw share of positive reviews is
unusable as a ranking on its own: **8 634 games sit at exactly 100%**, most of them on a handful
of reviews. The Wilson 95% lower bound answers the question actually being asked — *how good is
this game, given how little we know about it* — by pulling small samples towards the middle.

In [12]:
games = (games
    .withColumn("reviews", F.col("positive") + F.col("negative"))
    .withColumn("positive_ratio",
                F.when(F.col("positive") + F.col("negative") > 0,
                       F.col("positive") / (F.col("positive") + F.col("negative")))))

z, n, p = F.lit(1.96), F.col("reviews"), F.col("positive_ratio")
wilson_lower_bound = (p + z * z / (2 * n) - z * F.sqrt((p * (1 - p) + z * z / (4 * n)) / n)) / (1 + z * z / n)
games = games.withColumn("wilson_score", F.when(n > 0, wilson_lower_bound))

display(games.filter(F.col("positive_ratio") == 1)
             .select("name", "positive", "negative", "positive_ratio", "wilson_score")
             .orderBy("reviews").limit(5))

,name,positive,negative,positive_ratio,wilson_score
0,CrossTrix,1,0,1.0,0.206543
1,Anti-Grav Bamboo-copter,1,0,1.0,0.206543
2,De Profundis,1,0,1.0,0.206543
3,Kill Tiger,1,0,1.0,0.206543
4,The Truck Game,1,0,1.0,0.206543


### 2.9 Platforms, and a revenue proxy

The `platforms` struct becomes three flags and a count. Revenue is not in the dataset; the
closest available stand-in is **owners x list price**, which section 4.4 uses and section 7
takes apart.

In [13]:
games = (games
    .withColumn("windows", F.col("platforms.windows"))
    .withColumn("mac", F.col("platforms.mac"))
    .withColumn("linux", F.col("platforms.linux"))
    .withColumn("n_platforms", F.col("platforms.windows").cast("int")
                             + F.col("platforms.mac").cast("int")
                             + F.col("platforms.linux").cast("int"))
    .withColumn("revenue_proxy", F.col("owners_mid") * F.col("initial_price_usd")))

games = games.cache()
print(f"{games.count():,} games ready, {len(games.columns)} columns")

55,690 games ready, 48 columns


### 2.10 Cleaning report

In [14]:
report = spark.createDataFrame([
    ("2.1  drop type != 'game'",            games.count(),  raw.count() - games.count()),
    ("2.2  price strings -> USD",           games.count(),  games.filter(F.col("discount_pct") > 0).count()),
    ("2.3  release_date -> date",           games.count(),  games.filter(F.col("release_date_parsed").isNull()).count()),
    ("2.4  genre -> array",                 games.count(),  games.filter(F.col("n_genres") == 0).count()),
    ("2.4  languages -> array",             games.count(),  games.filter(F.col("n_languages") == 0).count()),
    ("2.5  publisher name normalised",      games.count(),  games.filter(F.col("publisher") != F.col("publisher_clean")).count()),
    ("2.6  owners bracket -> bounds",       games.count(),  games.filter(F.col("owners_min").isNull()).count()),
    ("2.7  required_age -> 0-21 or NULL",   games.count(),  games.filter(F.col("age_rating").isNull()).count()),
    ("2.8  reviews -> Wilson score",        games.count(),  games.filter(F.col("wilson_score").isNull()).count()),
], ["rule", "rows_in", "rows_affected_or_null"])

display(report)

,rule,rows_in,rows_affected_or_null
0,2.1 drop type != 'game',55690,1
1,2.2 price strings -> USD,55690,2518
2,2.3 release_date -> date,55690,99
3,2.4 genre -> array,55690,160
4,2.4 languages -> array,55690,10
5,2.5 publisher name normalised,55690,271
6,2.6 owners bracket -> bounds,55690,0
7,2.7 required_age -> 0-21 or NULL,55690,5
8,2.8 reviews -> Wilson score,55690,163


### 2.11 How success is measured, from here on

Nothing in this dataset is a sales figure. Two measures stand in, and every comparison in the
notebook reports both:

- **median review count** — continuous, and it separates the bottom of the catalogue where the
  owners bracket cannot;
- **breakout rate** — the share of games that reached at least 100 000 owners.

They are not independent: on the log scale, review count and owner midpoint correlate at
**0.76**, which is what justifies using reviews as a stand-in for reach at all.

In [15]:
display(games.filter(F.col("reviews") > 0).select(
    F.round(F.corr(F.log("reviews"), F.log(F.greatest("owners_mid", F.lit(1)))), 3).alias("corr_log_reviews_owners")))

def outcome(df, *group_by):
    """Volume and the two success measures, for any grouping."""
    return (df.groupBy(*group_by).agg(
        F.count("*").alias("games"),
        F.percentile_approx("reviews", 0.5).alias("median_reviews"),
        F.round(100 * F.avg((F.col("owners_min") >= 100000).cast("int")), 1).alias("breakout_pct"),
        F.round(F.avg("positive_ratio"), 3).alias("mean_positive_ratio")))

,corr_log_reviews_owners
0,0.761


---
# 3. The market

## 3.1 Who publishes on Steam

*Chart: bar, `publisher_clean` x `games`.*

In [16]:
by_publisher = (games.groupBy("publisher_clean")
    .agg(F.count("*").alias("games"),
         F.round(F.sum("revenue_proxy") / 1e6).alias("revenue_proxy_musd"))
    .filter(F.col("publisher_clean").isNotNull()))

display(by_publisher.orderBy(F.desc("games")).limit(20))

,publisher_clean,games,revenue_proxy_musd
0,Big Fish Games,423,49.0
1,8floor,202,11.0
2,SEGA,165,1864.0
3,Strategy First,151,58.0
4,Square Enix,141,1213.0
5,Choice of Games,140,8.0
6,Sekai Project,132,83.0
7,HH-Games,132,8.0
8,Ubisoft,128,3599.0
9,Laush Studio,126,15.0


**Big Fish Games** has released the most games — 423, casual hidden-object titles — ahead of
8floor (202) and SEGA (165). Ubisoft is tenth with 128.

That ranking says less than the shape behind it. Steam has **29 824 named publishers for
55 690 games** (134 of which carry no publisher and sit outside every table here), and the
concentration is the finding:

In [17]:
concentration = (by_publisher
    .withColumn("size", F.when(F.col("games") == 1, "1 game")
                         .when(F.col("games") <= 5, "2-5 games")
                         .when(F.col("games") <= 20, "6-20 games")
                         .otherwise("21+ games"))
    .groupBy("size").agg(F.count("*").alias("publishers"), F.sum("games").alias("games"))
    .withColumn("pct_of_catalogue", F.round(100 * F.col("games") / games.count(), 1)))

display(concentration.orderBy("publishers"))

,size,publishers,games,pct_of_catalogue
0,21+ games,195,9502,17.1
1,6-20 games,816,7984,14.3
2,2-5 games,5795,15052,27.0
3,1 game,23018,23018,41.3


**41% of the catalogue comes from publishers that have released exactly one game**, and the
twenty largest publishers together account for 5% of releases. Steam is not a market of a few
big houses — it is a very long tail with a handful of large firms on the end of it.

Rank the same publishers by the revenue proxy instead of by volume and a different set of names
appears, Ubisoft first among them.

*Chart: bar, `publisher_clean` x `revenue_proxy_musd`.*

In [18]:
display(by_publisher.filter(F.col("games") >= 5)
                    .withColumn("revenue_per_game_musd", F.round(F.col("revenue_proxy_musd") / F.col("games"), 1))
                    .orderBy(F.desc("revenue_proxy_musd")).limit(20))

,publisher_clean,games,revenue_proxy_musd,revenue_per_game_musd
0,Ubisoft,128,3599.0,28.1
1,Electronic Arts,88,3266.0,37.1
2,Valve,35,2400.0,68.6
3,Xbox Game Studios,47,2275.0,48.4
4,Bethesda Softworks,55,2210.0,40.2
5,Rockstar Games,19,1893.0,99.6
6,Paradox Interactive,69,1870.0,27.1
7,SEGA,165,1864.0,11.3
8,"CAPCOM Co., Ltd.",25,1712.0,68.5
9,Activision,51,1680.0,32.9


Volume and revenue are close to unrelated: Big Fish Games' 423 releases are worth less than
CD PROJEKT RED's 7. **For Ubisoft the competitor set is not "everyone on Steam"** — it is the
thirty-odd publishers in this table.

## 3.2 The release calendar

*Chart: bar, `release_year` x `games`.*

In [19]:
per_year = (games.filter(F.col("release_year").isNotNull())
                 .groupBy("release_year").agg(F.count("*").alias("games"))
                 .orderBy("release_year"))
display(per_year.filter(F.col("release_year") >= 2006))

,release_year,games
0,2006,61
1,2007,98
2,2008,159
3,2009,311
4,2010,288
5,2011,267
6,2012,345
7,2013,471
8,2014,1557
9,2015,2575


Releases grew every year to 2018, dipped in 2019 (6 968 against 7 678), then **rose through
Covid — 8 305 in 2020 and 8 823 in 2021**, the two largest years in the dataset. Whatever the
pandemic did to the industry, it did not slow the flow of new games on Steam; the only visible
dip is *before* it.

2022 shows 7 455, but the snapshot stops on 11 November, so that year covers ten and a half
months. At the 2022 daily pace it would have landed around 8 600 — flat against 2021, not down.

In [20]:
display(games.filter(F.col("release_year") == 2022)
             .groupBy("release_month").agg(F.count("*").alias("games")).orderBy("release_month"))

,release_month,games
0,1,683
1,2,652
2,3,817
3,4,626
4,5,771
5,6,677
6,7,708
7,8,759
8,9,783
9,10,818


Within the year, the question that matters for a launch is not how many games ship in a month
but how the games that ship in it do. Restricted to paid games from the seven complete years
2015-2021:

*Chart: combo — bars `games`, line `median_reviews`, on `release_month`.*

In [21]:
display(outcome(games.filter(F.col("release_year").between(2015, 2021) & ~F.col("is_free")), "release_month")
        .orderBy("release_month"))

,release_month,games,median_reviews,breakout_pct,mean_positive_ratio
0,1,2483,25,8.2,0.726
1,2,2786,27,9.2,0.730
2,3,3012,24,9.4,0.730
3,4,2976,26,9.2,0.737
4,5,2976,26,10.0,0.734
5,6,2732,24,9.3,0.731
6,7,3170,21,7.4,0.725
7,8,3370,23,9.2,0.730
8,9,3456,22,9.3,0.730
9,10,3652,22,8.5,0.741


**November and December are the worst months to launch in** — median 19 reviews and a 7.5%
breakout rate, against 26-27 reviews and 9-10% in February to May. October and November take the
most releases and return the least attention per release. The holiday window belongs to the
titles that can buy visibility in it.

**Decision — release window: February to May, and not November or December.**

## 3.3 Price

*Chart: bar, `price_usd` x `games`, on the 12 most common price points.*

In [22]:
display(games.filter(~F.col("is_free")).select(
    F.round(F.percentile_approx("price_usd", 0.5), 2).alias("median_paid"),
    F.round(F.percentile_approx("price_usd", 0.9), 2).alias("p90"),
    F.round(F.percentile_approx("price_usd", 0.99), 2).alias("p99")))

display(games.groupBy("price_usd").agg(F.count("*").alias("games")).orderBy(F.desc("games")).limit(12))

,median_paid,p90,p99
0,5.99,19.99,49.99


,price_usd,games
0,0.00,7779
1,4.99,6250
2,9.99,6126
3,0.99,5243
4,1.99,4193
5,2.99,3639
6,14.99,2878
7,19.99,2820
8,3.99,2530
9,6.99,1856


Steam prices are anchored on `.99`: after the 7 779 free games, the catalogue piles up on $4.99,
$9.99 and $0.99. The median paid game is **$5.99** and the 99th percentile is $49.99 — a $70
release is off this chart entirely.

*Chart: combo — bars `games`, line `breakout_pct`, on `band`.*

In [23]:
price_band = (F.when(F.col("is_free"), "0 free")
               .when(F.col("price_usd") < 5, "1 under $5")
               .when(F.col("price_usd") < 10, "2 $5-10")
               .when(F.col("price_usd") < 20, "3 $10-20")
               .when(F.col("price_usd") < 40, "4 $20-40")
               .otherwise("5 $40+"))

display(outcome(games.withColumn("band", price_band), "band").orderBy("band"))

,band,games,median_reviews,breakout_pct,mean_positive_ratio
0,0 free,7779,47,20.0,0.697
1,1 under $5,23478,15,5.9,0.724
2,2 $5-10,12450,23,9.0,0.754
3,3 $10-20,9022,79,18.1,0.768
4,4 $20-40,2394,378,31.9,0.770
5,5 $40+,567,344,32.5,0.751


Every step up the price ladder buys more of both measures: from 15 median reviews and a 5.9%
breakout rate under $5, to **378 reviews and 31.9% in the $20-40 band**. Above $40 the sample
thins to 567 games and stops improving.

The arrow does not point the way it looks. A studio does not become successful by charging $30 —
it charges $30 because it built something that can carry the price. What the table does say is
that the $20-40 band is **where games of Ubisoft's scale actually live**, and that pricing a
premium title below $20 puts it among games that are not competing for the same attention.

**Decision — price: $29.99 to $39.99.**

In [24]:
display(games.filter(F.col("discount_pct") > 0).select(
    F.count("*").alias("discounted_games"),
    F.round(100 * F.count("*") / games.count(), 1).alias("pct_of_catalogue"),
    F.round(F.avg("discount_pct"), 1).alias("mean_discount"),
    F.percentile_approx("discount_pct", 0.5).alias("median_discount")))

,discounted_games,pct_of_catalogue,mean_discount,median_discount
0,2518,4.5,57.6,60


Discounts are worth one line: **2 518 games, 4.5% of the catalogue, at a median 60% off**. But
this is a one-day snapshot, and Steam's sales are periodic — the figure measures the day the data
was pulled, not how often games go on sale. Nothing in this dataset can answer the second
question.

## 3.4 Languages

*Chart: bar, `language` x `games`.*

In [25]:
by_language = (games.select(F.explode("languages_list").alias("language"))
                    .groupBy("language").agg(F.count("*").alias("games")))
display(by_language.orderBy(F.desc("games")).limit(20))

,language,games
0,English,55116
1,German,14019
2,French,13426
3,Russian,12922
4,Simplified Chinese,12782
5,Spanish - Spain,12233
6,Japanese,10368
7,Italian,9304
8,Portuguese - Brazil,6750
9,Korean,6600


**English is on 99% of the catalogue** and is not a decision. The next tier is: German (14 019),
French (13 426), Russian (12 922), Simplified Chinese (12 782), Spanish (12 233), Japanese
(10 368), Italian (9 304). The real question is how far down that list to go.

*Chart: combo — bars `games`, line `breakout_pct`, on `languages`.*

In [26]:
language_band = (F.when(F.col("n_languages") <= 1, "1")
                  .when(F.col("n_languages") <= 4, "2-4")
                  .when(F.col("n_languages") <= 9, "5-9")
                  .when(F.col("n_languages") <= 14, "10-14")
                  .when(F.col("n_languages") <= 20, "15-20")
                  .otherwise("21+"))

display(outcome(games.withColumn("languages", language_band), "languages")
        .orderBy(F.desc("median_reviews")))

,languages,games,median_reviews,breakout_pct,mean_positive_ratio
0,15-20,762,360,39.0,0.784
1,10-14,3595,306,34.7,0.773
2,5-9,7377,133,24.7,0.755
3,2-4,13026,26,7.9,0.741
4,21+,1265,25,13.0,0.747
5,1,29665,16,7.0,0.724


Localisation tracks success up to about twenty languages — 16 median reviews and a 7% breakout
rate for English-only, **360 reviews and 39% at 15-20 languages** — and then collapses at 21+.

That collapse is not a finding, it is a cluster of shovelware. The 1 265 games listing 21 or more
languages are dominated by a handful of publishers pushing dozens of near-identical $0.99 titles,
each shipped with every language Steam offers:

In [27]:
display(games.filter(F.col("n_languages") >= 21)
             .groupBy("publisher_clean")
             .agg(F.count("*").alias("games"),
                  F.round(F.avg("price_usd"), 2).alias("mean_price"),
                  F.percentile_approx("reviews", 0.5).alias("median_reviews"))
             .orderBy(F.desc("games")).limit(10))

,publisher_clean,games,mean_price,median_reviews
0,Hede,47,9.69,15
1,Blender Games,38,1.99,21
2,Laush Studio,25,2.31,25
3,Cute Hannah's Games,24,1.99,5
4,Dnovel,22,4.37,19
5,My Label Game Studio,22,1.99,4
6,Quiet River,21,1.99,2467
7,AFBIK Studio,14,0.49,22
8,L. Stotch,14,1.14,635
9,Snkl Studio,14,2.04,30


Adding a language string costs nothing and proves nothing. The band that means something is
10-20, where the localisation is real work.

**Decision — languages: twelve — EN, DE, FR, RU, zh-Hans, ES, JA, IT, then KO, pt-BR, PL and
zh-Hant.**

## 3.5 Age restriction

In [28]:
display(games.groupBy("age_rating").agg(F.count("*").alias("games")).orderBy(F.desc("games")).limit(8))

,age_rating,games
0,0,55029
1,15,265
2,18,223
3,17,38
4,16,38
5,12,32
6,13,26
7,14,10


Read literally, **only 656 games out of 55 690 (1.2%) carry any age restriction at all**, and 301
are 16+ or over. That is not a description of Steam's catalogue, it is a description of a field
nobody fills in: the store gates mature content through its own content descriptors, and
`required_age` is a legacy attribute left at 0.

The community tags do carry the signal, so the brief's question is better answered with them.

*Chart: bar, `mature` x `breakout_pct`.*

In [29]:
MATURE_TAGS = ["Violent", "Gore", "Nudity", "Sexual Content", "NSFW", "Hentai", "Mature"]

games = games.withColumn("mature",
    F.size(F.array_intersect(F.map_keys("tags"), F.array(*[F.lit(t) for t in MATURE_TAGS]))) > 0)

display(outcome(games, "mature"))
print("mature-tagged games that also declare an age rating:",
      games.filter(F.col("mature") & (F.col("age_rating") > 0)).count(),
      "of", games.filter("mature").count())

,mature,games,median_reviews,breakout_pct,mean_positive_ratio
0,True,7103,71,19.6,0.720
1,False,48587,23,10.8,0.739


mature-tagged games that also declare an age rating: 388 of 7103


**7 103 games — 12.8% of the catalogue — carry mature content tags**, and only 388 of them
declare an age rating for it. They also do better than the rest: median 71 reviews against 23,
and a 19.6% breakout rate against 10.8%.

Caution on the direction: mature themes are concentrated in the kind of large action and RPG
titles that would outperform anyway. The table is not evidence that adding blood sells copies.
It is evidence that **a mature rating is not a commercial handicap on Steam**, which is the only
thing the decision needs.

**Decision — age rating: mature (17+/PEGI 18) is not a constraint on the concept.**

## 3.6 The best rated games

In [30]:
print("games with a 100% positive ratio:", games.filter(F.col("positive_ratio") == 1).count())
display(games.filter(F.col("positive_ratio") == 1)
             .select("name", "positive", "negative", "positive_ratio")
             .orderBy("reviews").limit(5))

games with a 100% positive ratio: 8634


,name,positive,negative,positive_ratio
0,CrossTrix,1,0,1.0
1,Anti-Grav Bamboo-copter,1,0,1.0
2,De Profundis,1,0,1.0
3,Kill Tiger,1,0,1.0
4,The Truck Game,1,0,1.0


Ranking on the raw ratio returns 8 634 games tied at 100%, most of them on one or two reviews.
The Wilson lower bound breaks the tie by asking how much evidence sits behind the score; with a
floor of 500 reviews it produces a list that means something.

In [31]:
display(games.filter(F.col("reviews") >= 500)
             .select("name", "publisher_clean", "reviews",
                     F.round("positive_ratio", 4).alias("positive_ratio"),
                     F.round("wilson_score", 4).alias("wilson_score"),
                     "release_year")
             .orderBy(F.desc("wilson_score")).limit(15))

,name,publisher_clean,reviews,positive_ratio,wilson_score,release_year
0,Flowers -Le volume sur ete-,JAST USA,938,0.9989,0.9940,2018
1,Aseprite,Igara Studio,11903,0.9933,0.9916,2016
2,A Short Hike,adamgryu,11732,0.9926,0.9909,2019
3,Senren＊Banka,"HIKARI FIELD, NekoNyan Ltd.",10677,0.9921,0.9903,2020
4,Aventura Copilului Albastru și Urât,Codrin Bradea,2217,0.9937,0.9894,2021
5,People Playground,Studio Minus,144569,0.9886,0.9880,2019
6,Portal 2,Valve,309441,0.9878,0.9874,2011
7,CULTIC,3D Realms,2037,0.9921,0.9873,2022
8,Vampire Survivors,poncle,131935,0.9877,0.9871,2022
9,Patrick's Parabox,Patrick Traynor,1500,0.9927,0.9869,2022


The top of the list is not made of blockbusters. *Aseprite* is a pixel-art editor, *A Short Hike*
and *Patrick's Parabox* are one-person indie games, and the highest-rated large title is
**Portal 2 at 98.8% over 309 441 reviews** — which is the more useful benchmark, because
sustaining that ratio at that volume is the hard part.

**Reference point — a Ubisoft-scale title doing well on Steam sits near 90% positive; 95%+ at
100 000+ reviews is exceptional.**

---
# 4. Genres

`genre` holds one to seven labels per game. Exploding it gives one row per (game, genre), so a
game counted under Action is also counted under RPG — the shares below add up to more than 100%
by construction.

In [32]:
# the raw comma-separated string is dropped: the exploded label replaces it
genre_rows = games.drop("genre").select("*", F.explode("genres").alias("genre")).cache()
print(f"{genre_rows.count():,} (game, genre) rows for {games.count():,} games")

157,110 (game, genre) rows for 55,690 games


## 4.1 What is on the shelf

*Chart: bar, `genre` x `games`.*

In [33]:
display(genre_rows.groupBy("genre").agg(F.count("*").alias("games"))
        .withColumn("pct_of_catalogue", F.round(100 * F.col("games") / games.count(), 1))
        .orderBy(F.desc("games")).limit(15))

,genre,games,pct_of_catalogue
0,Indie,39681,71.3
1,Action,23759,42.7
2,Casual,22086,39.7
3,Adventure,21431,38.5
4,Strategy,10895,19.6
5,Simulation,10836,19.5
6,RPG,9534,17.1
7,Early Access,6145,11.0
8,Free to Play,3393,6.1
9,Sports,2666,4.8


**Indie is on 71% of the catalogue** — and it is not a genre. Neither are *Early Access* (11%)
nor *Free to Play* (6%): they describe how a game is funded and sold, not what it is. Excluding
those three, the shelf is **Action (43%), Casual (40%), Adventure (38%), Strategy (20%),
Simulation (19%), RPG (17%)**, and everything else is under 5%.

For Ubisoft the labels that matter are the six real ones. The rest of section 4 keeps all of
them in the tables, because the contrast between *Indie* and the others is itself informative.

## 4.2 Which genres are liked

Restricted to the 21 669 games with at least 50 reviews, so the ratio means something.

*Chart: bar, `genre` x `median_positive_ratio`.*

In [34]:
display(genre_rows.filter(F.col("reviews") >= 50).groupBy("genre")
        .agg(F.count("*").alias("games"),
             F.round(F.percentile_approx("positive_ratio", 0.5), 3).alias("median_positive_ratio"),
             F.round(F.sum("positive") / (F.sum("positive") + F.sum("negative")), 3).alias("pooled_ratio"))
        .filter(F.col("games") >= 100).orderBy(F.desc("median_positive_ratio")))

,genre,games,median_positive_ratio,pooled_ratio
0,Casual,6847,0.818,0.870
1,Adventure,8766,0.817,0.841
2,Indie,14570,0.813,0.886
3,Design & Illustration,133,0.809,0.963
4,Animation & Modeling,121,0.806,0.965
5,Action,9190,0.797,0.850
6,RPG,4551,0.794,0.856
7,Utilities,216,0.784,0.947
8,Simulation,4857,0.783,0.867
9,Strategy,4704,0.781,0.849


The spread is narrow — **0.82 for Casual and Adventure down to 0.78 for Strategy** — with one
exception. **Massively Multiplayer sits at 0.665**, fifteen points below everything else, and it
is the genre where the median game is actively disliked. Live-service games are judged on
servers, monetisation and updates long after launch, and this is what that judgement looks like
in aggregate.

The two columns disagree on purpose. The pooled ratio weights every review equally, so it
measures *the average experience across the genre*; the median weights every game equally, so it
measures *the typical game*. Indie is 0.813 typical and 0.886 pooled — its big titles are much
better liked than its median one.

**Decision — avoid a live-service / MMO structure. The satisfaction penalty is the largest
single effect in the genre data.**

## 4.3 Do publishers have favourite genres

*Chart: stacked bar, `publisher_clean` x `games`, grouped by `genre`.*

In [35]:
top_publishers = [r[0] for r in by_publisher.orderBy(F.desc("games")).limit(8).collect()]

display(genre_rows.filter(F.col("publisher_clean").isin(top_publishers))
        .groupBy("publisher_clean", "genre").agg(F.count("*").alias("games"))
        .withColumn("rank", F.row_number().over(
            Window.partitionBy("publisher_clean").orderBy(F.desc("games"))))
        .filter(F.col("rank") <= 3).orderBy("publisher_clean", "rank"))

,publisher_clean,genre,games,rank
0,8floor,Casual,202,1
1,8floor,Strategy,22,2
2,8floor,Simulation,10,3
3,Big Fish Games,Casual,419,1
4,Big Fish Games,Adventure,393,2
5,Big Fish Games,Simulation,7,3
6,Choice of Games,RPG,139,1
7,Choice of Games,Indie,136,2
8,Choice of Games,Adventure,112,3
9,HH-Games,Casual,132,1


Emphatically yes, and the specialisation is near-total: **Big Fish Games is 419 Casual and 393
Adventure out of 423 games**, 8floor is 202 Casual out of 202, Strategy First is Strategy,
Choice of Games is RPG. These are not diversified catalogues — each of these publishers has one
formula and repeats it.

Ubisoft's own Steam catalogue is the same shape, around a different centre:

In [36]:
display(genre_rows.filter(F.col("publisher_clean").rlike("^Ubisoft"))
        .groupBy("genre").agg(F.count("*").alias("games"),
                              F.round(F.sum("revenue_proxy") / 1e6).alias("revenue_proxy_musd"))
        .orderBy(F.desc("games")).limit(8))

,genre,games,revenue_proxy_musd
0,Action,75,3379.0
1,Adventure,49,2298.0
2,Strategy,23,131.0
3,RPG,20,899.0
4,Simulation,18,78.0
5,Racing,14,251.0
6,Casual,13,86.0
7,Indie,7,5.0


**Action (75 titles) and Adventure (49)**, then Strategy and RPG. Whatever the next game
is, it will be read by players against that catalogue.

## 4.4 Which genres are worth entering

The revenue proxy is `owners x list price`. Summed by genre it gives each genre's share of the
market's value, which can be set against its share of releases:

**opportunity = share of proxy revenue / share of releases.** Above 1, the genre returns more
value than the shelf space it takes.

*Chart: scatter, `share_of_releases` x `share_of_revenue`, size `games`, label `genre`.*

In [37]:
total_games = games.count()
total_revenue = games.agg(F.sum("revenue_proxy")).collect()[0][0]

genre_economics = (genre_rows.groupBy("genre")
    .agg(F.count("*").alias("games"),
         F.sum("revenue_proxy").alias("revenue"),
         F.round(F.avg("revenue_proxy")).alias("mean_revenue_per_game"),
         F.round(100 * F.avg((F.col("owners_min") >= 100000).cast("int")), 1).alias("breakout_pct"),
         F.round(100 * F.avg(F.col("is_free").cast("int")), 1).alias("pct_free"))
    .filter(F.col("games") >= 150)
    .withColumn("share_of_releases", F.round(100 * F.col("games") / total_games, 1))
    .withColumn("share_of_revenue", F.round(100 * F.col("revenue") / total_revenue, 1))
    .withColumn("opportunity", F.round((F.col("revenue") / total_revenue) / (F.col("games") / total_games), 2)))

display(genre_economics.drop("revenue").orderBy(F.desc("opportunity")))

,genre,games,mean_revenue_per_game,breakout_pct,pct_free,share_of_releases,share_of_revenue,opportunity
0,Massively Multiplayer,1460,5135848.0,32.3,53.0,2.6,7.8,2.96
1,RPG,9534,3136954.0,16.5,14.6,17.1,30.9,1.81
2,Action,23759,2630520.0,13.6,13.4,42.7,64.6,1.52
3,Strategy,10895,1924422.0,15.3,13.9,19.6,21.7,1.11
4,Adventure,21431,1860164.0,11.9,11.3,38.5,41.2,1.07
5,Simulation,10836,1797926.0,12.8,12.8,19.5,20.2,1.04
6,Racing,2155,1282033.0,11.9,12.7,3.9,2.9,0.74
7,Sports,2666,1190963.0,10.4,15.5,4.8,3.3,0.69
8,Early Access,6145,903136.0,7.6,17.9,11.0,5.7,0.52
9,Indie,39681,856837.0,9.7,12.4,71.3,35.2,0.49


Three genres return more than they take — **Massively Multiplayer at 2.96, RPG at 1.81, Action
at 1.52** — and two are badly crowded: **Casual takes 40% of the shelf for 8.7% of the value
(0.22), Indie 71% for 35% (0.49)**. Strategy, Adventure and Simulation sit at parity.

Two readings to be careful with. **Free to Play scores 0.02 not because free games make no money
but because the proxy cannot see any revenue that is not a store price** — an in-game economy is
invisible here. The same blindness deflates Massively Multiplayer, 53% of which is free, and it
still comes out on top; its true opportunity is higher than 2.96, not lower. And the whole column
inherits the proxy's biases, so it ranks genres, it does not value them.

Set against 4.2, that gives one clean answer: **MMO buys the best economics and the worst
satisfaction. RPG is second on economics with no such penalty.**

## 4.5 Is any genre emerging

*Chart: grouped bar, `genre` x `pct`, grouped by `release_year`.*

In [38]:
mix = (genre_rows.filter(F.col("release_year").isin(2017, 2022))
       .groupBy("release_year", "genre").agg(F.count("*").alias("games")))
year_totals = mix.groupBy("release_year").agg(F.sum("games").alias("total"))

genre_mix = (mix.join(year_totals, "release_year")
    .withColumn("pct", F.round(100 * F.col("games") / F.col("total"), 1))
    .groupBy("genre").pivot("release_year", [2017, 2022]).agg(F.first("pct"))
    .withColumnRenamed("2017", "pct_2017").withColumnRenamed("2022", "pct_2022")
    .withColumn("shift_pts", F.round(F.col("pct_2022") - F.col("pct_2017"), 1)))

display(genre_mix.orderBy(F.desc("pct_2022")).limit(12))

,genre,pct_2017,pct_2022,shift_pts
0,Indie,25.1,24.7,-0.4
1,Action,15.6,14.7,-0.9
2,Adventure,13.5,14.5,1.0
3,Casual,14.0,14.5,0.5
4,Strategy,6.4,7.2,0.8
5,Simulation,6.7,7.1,0.4
6,RPG,5.0,6.8,1.8
7,Early Access,3.5,5.7,2.2
8,Sports,2.0,1.5,-0.5
9,Racing,1.3,1.5,0.2


Five years apart, the mix barely moves: Indie 25.1% → 24.7%, Action 15.6% → 14.7%, Casual
14.0% → 14.5%. The only shifts worth naming are **Early Access, 3.5% → 5.7%**, and **Free to Play
collapsing from 2.3% to 0.4%** — the latter partly a labelling change, since free games kept
their share of the catalogue while the genre tag stopped being applied.

**No genre is emerging.** A concept does not need to catch a wave here, because there isn't one;
it needs to be good in a category that already pays.

## 4.6 The slot

Putting 4.2 and 4.4 together with the price band from 3.3 — games released from 2018, priced
$20-40:

*Chart: combo — bars `games`, line `breakout_pct`, on `genre`.*

In [39]:
display(outcome(genre_rows.filter((F.col("release_year") >= 2018)
                                  & F.col("price_usd").between(20, 40)), "genre")
        .filter(F.col("games") >= 150).orderBy(F.desc("breakout_pct")))

,genre,games,median_reviews,breakout_pct,mean_positive_ratio
0,Action,776,622,34.9,0.753
1,RPG,471,484,33.8,0.766
2,Strategy,474,431,31.2,0.763
3,Simulation,551,468,29.2,0.764
4,Indie,866,266,27.9,0.783
5,Adventure,745,348,27.8,0.790
6,Early Access,242,273,27.7,0.764
7,Casual,315,77,15.2,0.792


In the band Ubisoft would actually price into, **Action reaches a 34.9% breakout rate over 776
games and RPG 33.8% over 471**, against 27.8% for Adventure. Adventure trades six points of
breakout for the best satisfaction of the three (0.790). Action and RPG overlap heavily — most of these games carry both labels —
which is the combination the recommendation lands on.

**Decision — genre: Action-RPG, single-player, premium. Not Casual, not Indie-positioned, not
live service.**

---
# 5. Platforms

## 5.1 What Steam runs on

*Chart: bar, `platform` x `games`.*

In [40]:
display(games.select(
    F.sum(F.col("windows").cast("int")).alias("windows"),
    F.sum(F.col("mac").cast("int")).alias("mac"),
    F.sum(F.col("linux").cast("int")).alias("linux")))

display(games.groupBy("windows", "mac", "linux").agg(F.count("*").alias("games"))
        .orderBy(F.desc("games")))

,windows,mac,linux
0,55675,12769,8457


,windows,mac,linux,games
0,True,False,False,41271
1,True,True,True,6806
2,True,True,False,5951
3,True,False,True,1647
4,False,True,False,11
5,False,False,True,3
6,False,True,True,1


**Windows is not a choice: 55 675 of 55 690 games support it, and the 15 that do not are
curiosities.** Mac reaches 22.9% of the catalogue and Linux 15.2%, and they travel together —
6 806 games ship all three, more than the 5 951 that add Mac alone.

So the platform question is only ever "do we port", never "which one".

## 5.2 Which genres get ported

*Chart: bar, `genre` x `pct_mac` and `pct_linux`.*

In [41]:
display(genre_rows.groupBy("genre").agg(
            F.count("*").alias("games"),
            F.round(100 * F.avg(F.col("mac").cast("int")), 1).alias("pct_mac"),
            F.round(100 * F.avg(F.col("linux").cast("int")), 1).alias("pct_linux"))
        .filter(F.col("games") >= 150).orderBy(F.desc("pct_mac")))

,genre,games,pct_mac,pct_linux
0,Game Development,159,32.7,22.0
1,Strategy,10895,27.6,16.8
2,Indie,39681,25.0,17.6
3,Free to Play,3393,24.9,14.0
4,Design & Illustration,406,24.6,13.3
5,RPG,9534,23.6,16.0
6,Adventure,21431,23.5,15.4
7,Casual,22086,23.2,15.0
8,Animation & Modeling,322,23.0,11.8
9,Simulation,10836,22.5,14.1


The spread runs from **Strategy at 27.6% Mac and 16.8% Linux down to Early Access at 14.6% and
10.3%**, with Action near the bottom of the range at 19.2% and 14.2%. Turn-based and 2D-heavy
categories port more; action games, which lean hardest on the graphics stack, port least. The
technical cost of the port is visible in the genre mix.

Note the two ends: Strategy is 8 points above Action on Mac, but no genre is anywhere near
Windows' 99.97%.

## 5.3 Does porting pay

*Chart: combo — bars `games`, line `breakout_pct`, on `n_platforms`.*

In [42]:
display(outcome(games, "n_platforms").orderBy("n_platforms"))

,n_platforms,games,median_reviews,breakout_pct,mean_positive_ratio
0,1,41285,21,10.0,0.722
1,2,7599,37,14.0,0.773
2,3,6806,79,21.2,0.783


Three platforms beats one on both measures — 79 median reviews against 21, a 21.2% breakout rate
against 10.0%. That comparison is worthless on its own: a studio that expects a hit is exactly
the studio that pays for ports, so the gap could be entirely selection.

Two checks. First, hold the price band and the release era fixed:

In [43]:
display(outcome(games.filter(F.col("release_year") >= 2018)
                     .withColumn("band", price_band)
                     .withColumn("ported", F.col("n_platforms") > 1),
                "band", "ported").orderBy("band", "ported"))

,band,ported,games,median_reviews,breakout_pct,mean_positive_ratio
0,0 free,False,4007,28,12.7,0.697
1,0 free,True,1219,48,18.0,0.724
2,1 under $5,False,13769,9,2.0,0.728
3,1 under $5,True,3315,13,2.9,0.796
4,2 $5-10,False,6145,12,2.6,0.752
5,2 $5-10,True,1970,21,4.8,0.813
6,3 $10-20,False,4911,38,8.9,0.758
7,3 $10-20,True,1610,87,16.3,0.833
8,4 $20-40,False,1525,238,25.3,0.764
9,4 $20-40,True,318,494,31.4,0.809


The gap survives in five of the six bands: at $10-20, 87 median reviews ported against 38; at
$20-40, 494 against 238 and a 31.4% breakout rate against 25.3%. It reverses only above $40, on
48 ported games — too few to read. Second, if porting were a big-publisher behaviour, porting
rates would climb with catalogue size — they do not:

In [44]:
publisher_size = by_publisher.select("publisher_clean", F.col("games").alias("publisher_games"))

display(games.join(publisher_size, "publisher_clean")
        .withColumn("publisher_size", F.when(F.col("publisher_games") == 1, "1 game")
                                       .when(F.col("publisher_games") <= 5, "2-5 games")
                                       .otherwise("6+ games"))
        .groupBy("publisher_size").agg(
            F.count("*").alias("games"),
            F.round(100 * F.avg((F.col("n_platforms") > 1).cast("int")), 1).alias("pct_ported")))

,publisher_size,games,pct_ported
0,2-5 games,15052,27.8
1,6+ games,17486,26.6
2,1 game,23018,24.0


**24.0% for one-game publishers, 27.8% for small ones, 26.6% for large ones** — flat. Porting is
not something only big studios do, which removes the most obvious confounder without removing
the causality problem: within any size class, the games that get ported are still the ones
someone believed in.

What survives is a floor, not a lift: the association is consistent, and the Linux port also
covers the Steam Deck, which shipped in February 2022 and is not yet visible in this snapshot.

**Decision — platforms: Windows at launch, Mac and Linux planned in. Linux is the Steam Deck
route and the cheaper of the two to add once the engine is portable.**

---
# 6. The brief for the next game

Six decisions, each from the section that produced it.

| | decision | evidence |
|---|---|---|
| **Genre** | Action-RPG, single-player, premium | Action 1.52 and RPG 1.81 on revenue-to-releases; 34.9% and 33.8% breakout in the $20-40 band (4.4, 4.6) |
| **Not** | live service or MMO | MMO carries the best economics *and* a 0.665 median positive ratio, fifteen points below every other genre (4.2) |
| **Price** | $29.99 - $39.99 | the $20-40 band returns 378 median reviews and a 31.9% breakout rate; above $40 the market thins to 567 games (3.3) |
| **Platforms** | Windows at launch, Mac and Linux planned in | Windows is 99.97% of the catalogue; ported games lead inside every price band, and porting is not a big-studio behaviour (5.1, 5.3) |
| **Languages** | twelve: EN, DE, FR, RU, zh-Hans, ES, JA, IT, KO, pt-BR, PL, zh-Hant | success climbs to the 15-20 band, 39% breakout, then collapses into shovelware above 21 (3.4) |
| **Window** | February to May | November and December take the most releases and return the least: 19 median reviews and 7.5% breakout, against 26-27 and 9-10% in spring (3.2) |
| **Rating** | mature is not a constraint | 12.8% of the catalogue is mature-tagged and outperforms the rest; Steam's own age field is empty for 98.8% of games (3.5) |

Two things this study says about the field the game will land in, beyond the product itself:

- **The competitor set is about thirty publishers, not fifty thousand.** 41% of Steam's catalogue
  comes from publishers with a single release, and the twenty largest account for 5% of it.
  Ubisoft's rivals are the names in 3.1's revenue table.
- **There is no wave to catch.** The genre mix moved by less than two points in five years. The
  concept has to win on execution inside a category that already pays, not on timing.

The quality bar is section 3.6's: **90% positive is a good Ubisoft-scale result on Steam, and 95%
at scale is exceptional** — Portal 2 territory.

---
# 7. What this dataset cannot decide

**It stops on 11 November 2022.** Every 2022 figure covers ten and a half months, and nothing
after that date exists — no Steam Deck effect, no post-2022 pricing.

**There are no sales.** `owners` is SteamSpy's *estimate*, published as a bracket, and 68% of the
catalogue falls in the bottom one. The revenue proxy multiplies that bracket's midpoint by the
list price, so it ignores Steam's 30% cut, regional pricing, discounts, refunds, bundles, free
keys — and it values every free-to-play game at zero, which is why section 4.4 reads *Free to
Play: 0.02* for a business model that funds some of the largest games in the table.

**Reviews are not players.** They correlate with owners at 0.76 on the log scale, which is enough
to rank and not enough to size.

**Nothing here is causal.** Price, ports and localisation are all things a studio chooses because
it already expects the game to sell. Section 5.3 removes the most obvious confounder for porting
and cannot remove the rest. Every decision in section 6 is a reading of where successful games
are, not a recipe for becoming one.

**Delisted games are absent.** The catalogue is what was on sale in November 2022, so failures
that were pulled never appear — the breakout rates are, if anything, optimistic.

**One store, one region.** Steam is not consoles, not mobile, not the Epic store, and the prices
are US dollars.